# Practice 02 · Control basics with MuJoCo

Run the cells from top to bottom. This notebook contains all model, simulation, rendering, and control functions. No local `forc` package or repository layout is required.

**Environment:** Python with `numpy`, `scipy`, `mujoco==3.13.0`, `matplotlib`, `mediapy`, and a Jupyter kernel; FFmpeg and an OpenGL backend are needed for videos. In the course repository, use the `p02` install profile. On headless Linux, choose `MUJOCO_GL=egl` or `MUJOCO_GL=osmesa` before starting the kernel, according to the installed driver.

**Outputs:** plots and playable videos appear under their simulation cells. They are computed when you run the notebook, not loaded from media files. Clear all outputs before committing this source notebook.

Set `VISUALIZE = False` in the first code cell for numeric-only execution. Leave it enabled for inline plots and MuJoCo videos.


## 1. State, input, and the simulation loop

A model tells us how a state changes. A controller chooses an input from measurements and a reference. Keep these roles separate: a controller that works on an approximate model must still be tested on the actual plant.

For a continuous-time linear time-invariant (LTI) system,

$$
\dot{\mathbf{x}}=A\mathbf{x}+B\mathbf{u},
\qquad
\mathbf{y}=C\mathbf{x}+D\mathbf{u}.
$$

- **State:** $\mathbf{x}\in\mathbb{R}^{n}$ contains enough information to predict the future given the input.
- **Input:** $\mathbf{u}\in\mathbb{R}^{m}$ is the force or torque we command.
- **Output:** $\mathbf{y}\in\mathbb{R}^{p}$ is what we measure or plot; it need not contain the whole state.
- **Dimensions:** $A\in\mathbb{R}^{n\times n}$, $B\in\mathbb{R}^{n\times m}$, $C\in\mathbb{R}^{p\times n}$, and $D\in\mathbb{R}^{p\times m}$. A single-input, single-output model has a **scalar** direct-feedthrough term, represented as a $1\times1$ matrix.



### MuJoCo conventions used here

Every model in this practice uses only scalar slide or hinge joints. Therefore, and **only for these models**, we can concatenate positions and velocities directly:

$$
\mathbf{x}=\begin{bmatrix}\mathbf{q}\\\mathbf{v}\end{bmatrix},
\qquad
\mathbf{q}=\texttt{data.qpos},\quad
\mathbf{v}=\texttt{data.qvel},\quad
\mathbf{u}=\texttt{data.ctrl}.
$$

Each actuator is a direct-drive `<motor>` with unit gear and no actuator activation state. Its control is a force in N for a slide joint, or torque in N m for a hinge joint. This interpretation is **not universal**: a position actuator would interpret `ctrl` differently. Floating-base robots also have quaternion coordinates, so revisit the configuration/velocity distinction from practice 01 before generalizing these formulas.

The models defined below specify inertia explicitly, disable collisions, and omit friction and joint limits. Springs and viscous dampers are real MuJoCo passive forces. These choices isolate the equations we are studying; a finite rail, impacts, backlash, or electrical dynamics would require additional modeling.

### Load the tools

This notebook contains every model and helper function used in the experiments. You only need the installed Python packages; no course-package imports, repository paths, robot downloads, or external Python files are required. The conceptual diagrams are embedded in the notebook as attachments.

The first cell imports the libraries and sets a shared visual style. Plots are displayed directly, and `mediapy` embeds encoded video in the output of the cell. No plot or video files are saved in the course tree.

In [ ]:
from dataclasses import dataclass
import numpy as np
import matplotlib.pyplot as plt
import mujoco
import mediapy as media
from scipy.integrate import solve_ivp
from scipy.signal import step, cont2discrete, place_poles

# Set False for a numeric-only run without an OpenGL context.
VISUALIZE = True
DT, DURATION = 0.002, 10.0
INK, BLUE, TEAL, ORANGE, GRAY = "#172c46", "#145c98", "#267b77", "#bf5b19", "#687786"
COLORS = [BLUE, TEAL, ORANGE, GRAY]
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=COLORS),
    "font.family": "DejaVu Sans", "font.size": 11,
    "text.color": INK, "axes.labelcolor": INK,
    "xtick.color": GRAY, "ytick.color": GRAY,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c9d5e0", "axes.grid": True,
    "grid.color": "#e2e9f0", "grid.linewidth": 0.7,
    "lines.linewidth": 2.2, "figure.facecolor": "white",
    "figure.constrained_layout.use": True,
})
print("MuJoCo", mujoco.__version__, "| physics and control:", 1 / DT, "Hz")

### What happens at each simulation step?

The controller runs at **500 Hz**. At sample $k$, we read the state, calculate the input, apply any force limit, then advance MuJoCo by $h=0.002$ s:

$$
\mathbf{u}_k=\operatorname{clip}\!\left(\pi(\mathbf{x}_k),-\mathbf{u}_{\max},\mathbf{u}_{\max}\right).
$$

`rollout` writes this input to `data.ctrl` and calls `mj_step`. The RK4 integrator evaluates the plant internally while **holding this sampled input fixed**. Thus the implementation is sampled feedback, even though the design equations are continuous-time. The trace contains the initial and final state, requested input, applied input, and simulation time. With no controller, the helper applies zero input.

For fair comparisons, `linear_rollout` uses the exact zero-order-hold discretization of the linear plant, with the same sample period and force limit. It does not use an unrelated integration grid.

### Implement the step loop here

`Trace` keeps states and inputs in memory. `state` and `set_state` make the coordinate order explicit. The controller is called once per step, and its command is held fixed during that step. A fresh `MjData` gives each experiment an independent initial condition.

In [ ]:
@dataclass
class Trace:
    time: np.ndarray
    x: np.ndarray
    requested: np.ndarray
    applied: np.ndarray
    model: object
    label: str

def state(model, data):
    return np.r_[data.qpos.copy(), data.qvel.copy()]

def set_state(model, data, x):
    x = np.asarray(x, float)
    if x.shape != (model.nq + model.nv,):
        raise ValueError('Expected x = [qpos, qvel]')
    data.qpos[:] = x[:model.nq]
    data.qvel[:] = x[model.nq:]

def rollout(model, initial, controller=None, *, duration=8.0, limit=np.inf, label='MuJoCo'):
    d = mujoco.MjData(model)
    set_state(model, d, initial)
    n = round(duration / model.opt.timestep)
    rows = []
    requested = []
    applied = []
    times = []
    for i in range(n + 1):
        u = np.zeros(model.nu) if controller is None else np.atleast_1d(controller(model, d)).astype(float)
        command = np.clip(u, -limit, limit)
        d.ctrl[:] = command
        mujoco.mj_forward(model, d)
        times.append(i * model.opt.timestep)
        rows.append(state(model, d))
        requested.append(u.copy())
        applied.append(command.copy())
        if not np.isfinite(rows[-1]).all():
            raise ValueError('Non-finite simulation state')
        if i < n:
            mujoco.mj_step(model, d)
    return Trace(np.array(times), np.array(rows), np.array(requested), np.array(applied), model, label)

## 2. Mass–spring–damper: a linear plant

A translating mass provides the shortest route from Newton's law to a state-space model. Position $q$ is measured from the spring's relaxed position; positive force and displacement point to the right.



### Build the model and its renderer

The shared MJCF wrapper below defines only the scene and numerical settings. Each later section supplies its own bodies, joints, inertias, and actuators in a visible constructor. `MjModel.from_xml_string` compiles that XML; no model file is loaded from disk.

In [ ]:
def xml_model(body, *, dt=0.002, gravity='0 0 0', actuator='', tendon='', static=''):
    """Explicit inertias and zero collision forces keep the teaching equations exact."""
    return f'''<mujoco model="FORC control basics">
      <compiler angle="radian"/>
      <option timestep="{dt}" gravity="{gravity}" integrator="RK4"/>
      <visual><global offwidth="800" offheight="450"/><headlight ambient=".4 .4 .4" diffuse=".6 .6 .6"/></visual>
      <asset><texture type="skybox" builtin="gradient" rgb1=".92 .95 .98" rgb2=".98 .99 1" width="128" height="768"/></asset>
      <default><geom contype="0" conaffinity="0" rgba=".0784 .3608 .5961 1"/><joint damping="0" armature="0"/></default>
      <worldbody><light pos="0 -3 5" dir="0 0 -1"/>
        <geom type="plane" size="8 8 .05" rgba=".94 .96 .98 1"/>
        {static}{body}
      </worldbody>{tendon}<actuator>{actuator}</actuator>
    </mujoco>'''

def mass_spring(*, mass=1., damping=2., stiffness=5., dt=0.002):
    body=f'''<body name="mass" pos="0 0 .3">
      <joint name="position" type="slide" axis="1 0 0" stiffness="{stiffness}" damping="{damping}"/>
      <inertial pos="0 0 0" mass="{mass}" diaginertia=".04 .04 .04"/>
      <geom type="box" size=".22 .2 .2"/><site name="attachment" pos="-.22 0 0"/>
    </body>'''
    return mujoco.MjModel.from_xml_string(xml_model(body,dt=dt,
        actuator='<motor joint="position" gear="1"/>',
        static='<geom type="box" pos="-1.6 0 .4" size=".04 .3 .4" rgba=".09 .17 .27 1"/>'))

The camera choices let us see translation from the side and rotor angles from above. `decorate` draws the visible spring; its restoring force still comes from the joint's `stiffness`. These are drawing operations, separate from the equations of motion.

In [ ]:
def camera_for(kind):
    cam = mujoco.MjvCamera()
    cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    cam.azimuth = 90
    cam.elevation = -25
    cam.distance = 4.0
    cam.lookat[:] = [0, 0, 0.3]
    if kind in {'rotor', 'flexible'}:
        cam.elevation = -80
        cam.distance = 3.3 if kind == 'flexible' else 2.4
        cam.lookat[:] = [0, 0, 0.2]
    if kind == 'field':
        cam.elevation = -89
        cam.distance = 7.0
        cam.lookat[:] = [0.3, 0.4, 0]
    if kind == 'cart':
        cam.distance = 2.5
        cam.elevation = -15
    return cam

def decorate(renderer, model, data, kind):
    if kind != 'spring':
        return
    points = np.column_stack([np.linspace(-1.55, data.qpos[0] - 0.22, 25), np.zeros(25), 0.3 + 0.045 * np.sin(np.linspace(0, 12 * np.pi, 25))])
    for a, b in zip(points[:-1], points[1:]):
        scene = renderer.scene
        if scene.ngeom >= scene.maxgeom:
            break
        geom = scene.geoms[scene.ngeom]
        mujoco.mjv_initGeom(geom, mujoco.mjtGeom.mjGEOM_CAPSULE, np.zeros(3), np.zeros(3), np.eye(3).ravel(), np.array([0.15, 0.48, 0.47, 1]))
        mujoco.mjv_connector(geom, mujoco.mjtGeom.mjGEOM_CAPSULE, 0.012, a, b)
        scene.ngeom += 1

`preview` sets the initial coordinates and calls `mj_forward` to update geometry **without advancing time**. It renders one image directly into the notebook. Every new mechanism is previewed before calculations.

In [ ]:
def preview(model, initial, kind):
    if not VISUALIZE:
        return
    data = mujoco.MjData(model)
    set_state(model, data, initial)
    mujoco.mj_forward(model, data)
    print(f"nq={model.nq}, nv={model.nv}, nu={model.nu}")
    with mujoco.Renderer(model, height=360, width=640) as renderer:
        renderer.update_scene(data, camera_for(kind))
        decorate(renderer, model, data, kind)
        frame = renderer.render().copy()
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.imshow(frame)
    ax.axis("off")
    plt.show()

First load and inspect the model. The wall and drawn spring identify the mechanism; the actual spring and damping forces are configured on the slide joint.

In [ ]:
spring = mass_spring(mass=1.0, damping=2.0, stiffness=5.0, dt=DT)
preview(spring, [1.0, 1.0], "spring")

Newton's law gives

$$
m\ddot q+b\dot q+kq=u.
$$

With $\mathbf{x}=[q,\dot q]^\mathsf{T}$ and measured output $y=q$,

$$
A=\begin{bmatrix}0&1\\-k/m&-b/m\end{bmatrix},\quad
B=\begin{bmatrix}0\\1/m\end{bmatrix},\quad
C=\begin{bmatrix}1&0\end{bmatrix},\quad D=\begin{bmatrix}0\end{bmatrix}.
$$

Use $m=1$ kg, $b=2$ N s/m, and $k=5$ N/m. The poles are $-1\pm2j$, so the unforced response oscillates while decaying. The equilibrium under a constant force is $q_\star=u/k$.

### Verify the equations before simulating

`derivative` assigns a state and input, calls `mj_forward`, and reads `qacc`. Compare that acceleration with the analytic state-space equation at a nonzero test state. This catches sign, mass, and actuator-gear mistakes before feedback hides them.

In [ ]:
def derivative(model, x, u):
    d = mujoco.MjData(model)
    set_state(model, d, x)
    d.ctrl[:] = np.atleast_1d(u)
    mujoco.mj_forward(model, d)
    return np.r_[d.qvel, d.qacc]

In [ ]:
A = np.array([[0., 1.], [-5., -2.]])
B = np.array([[0.], [1.]])
C, D = np.array([[1., 0.]]), np.zeros((1, 1))
x_test, u_test = np.array([0.3, -0.4]), np.array([1.7])
np.testing.assert_allclose(derivative(spring, x_test, u_test), A @ x_test + B @ u_test)
print("Open-loop poles:", np.linalg.eigvals(A))

### Display each experiment where it is run

`simulation_frames` reconstructs the computed states for the renderer at 25 frames/s. It yields frames one at a time, so we do not keep an entire uncompressed movie in memory. The cart camera follows its position; the position plot remains the quantitative record of translation.

`show_run` draws each state with its unit, then embeds a playable video immediately below the plot. `media.show_video` uses temporary encoding internally and returns an embedded notebook output; this function never writes a named video or plot file. Rendering does not advance the simulation or change the controller rate.

In [ ]:
def simulation_frames(trace, kind, fps=25):
    model = trace.model
    data = mujoco.MjData(model)
    camera = camera_for(kind)
    sample_times = np.arange(0, trace.time[-1] + 1e-10, 1 / fps)
    indices = np.rint(sample_times / model.opt.timestep).astype(int)
    with mujoco.Renderer(model, height=360, width=640) as renderer:
        for index in indices:
            set_state(model, data, trace.x[min(index, len(trace.x) - 1)])
            mujoco.mj_forward(model, data)
            if kind == "cart":
                camera.lookat[0] = data.qpos[0]
            renderer.update_scene(data, camera)
            decorate(renderer, model, data, kind)
            yield renderer.render().copy()


def show_run(trace, kind, name, labels, reference=None):
    if not VISUALIZE:
        return
    fig, axes = plt.subplots(len(labels), 1, figsize=(8, 2.1 * len(labels)),
                             sharex=True, squeeze=False)
    fig.suptitle(name.replace("-", " ").capitalize(), weight="bold", color=INK)
    for i, (quantity, unit) in enumerate(labels):
        ax = axes[i, 0]
        ax.plot(trace.time, trace.x[:, i], label=trace.label)
        if reference is not None:
            ax.plot(trace.time, reference[:, i], "--", color=TEAL,
                    label="Sampled linear model")
        ax.set_ylabel(f"{quantity} ({unit})")
        ax.legend(frameon=False)
    axes[-1, 0].set_xlabel("Time (s)")
    plt.show()
    media.show_video(simulation_frames(trace, kind), fps=25,
                     autoplay=False, loop=False)

### Release the mass, then apply a force step

The first experiment starts at $q(0)=1$ m and $\dot q(0)=1$ m/s with zero input. The second starts at rest and applies 2 N. Predict the final position in each case before running the cell.

In [ ]:
spring_free = rollout(spring, [1., 1.], duration=DURATION, label="Unforced MuJoCo")
show_run(spring_free, "spring", "spring-free", [("Position", "m"), ("Velocity", "m/s")])
spring_forced = rollout(spring, [0., 0.], lambda m, d: [2.], duration=DURATION, label="2 N step")
show_run(spring_forced, "spring", "spring-forced", [("Position", "m"), ("Velocity", "m/s")])

**Read the result:** the unforced response approaches zero; the forced response approaches 0.4 m. Without feedback, the spring determines the steady displacement. Increasing damping changes the transient but not this static equilibrium.

## 3. Motor control and second-order response

We now remove the spring and study a rotating inertia. This is a **torque-controlled motor model**, equivalent to assuming a sufficiently fast inner current loop. It does not model voltage, resistance, inductance, back EMF, or current saturation.



Load the rotor before deriving its response. Its colored radial marker makes the angle visible.

The constructor below assigns the rotational inertia explicitly and uses one hinge with viscous damping. Its unit-gear motor converts the control directly to torque.

In [ ]:
def rotor(*, inertia=1., damping=2., dt=0.002):
    body=f'''<body name="rotor" pos="0 0 .2">
      <joint name="angle" type="hinge" axis="0 0 1" damping="{damping}"/>
      <inertial pos="0 0 0" mass="8" diaginertia="{0.6*inertia} {0.6*inertia} {inertia}"/>
      <geom type="cylinder" size=".45 .06"/>
      <geom type="capsule" fromto="0 0 .07 .4 0 .07" size=".025" rgba=".75 .36 .1 1"/>
    </body>'''
    return mujoco.MjModel.from_xml_string(xml_model(body,dt=dt,actuator='<motor joint="angle" gear="1"/>'))

In [ ]:
motor = rotor(inertia=1.0, damping=2.0, dt=DT)
preview(motor, [1., 1.], "rotor")

The mechanical dynamics are

$$
J\ddot\theta+b\dot\theta=u,\qquad
A=\begin{bmatrix}0&1\\0&-b/J\end{bmatrix},\qquad
B=\begin{bmatrix}0\\1/J\end{bmatrix}.
$$

With $J=1$ kg m$^2$ and $b=2$ N m s/rad, the open-loop poles are 0 and $-2$. Velocity decays, but position is not restored to a unique angle. Releasing the rotor from $[1,1]^\mathsf{T}$ should leave it near 1.5 rad.

In [ ]:
motor_free = rollout(motor, [1., 1.], duration=DURATION, label="Zero torque")
show_run(motor_free, "rotor", "motor-free", [("Angle", "rad"), ("Angular velocity", "rad/s")])

### Choose the closed-loop response with PD feedback

For a constant target $r$, apply proportional position feedback and derivative velocity feedback:

$$
u=k_p(r-\theta)-k_d\dot\theta.
$$

For the error $e=\theta-r$,

$$
J\ddot e+(b+k_d)\dot e+k_p e=0.
$$

Compare its characteristic polynomial with the standard second-order form:

$$
s^2+2\zeta\omega_n s+\omega_n^2=0,
\qquad
k_p=J\omega_n^2,\quad k_d=2J\zeta\omega_n-b.
$$

- $\omega_n$ is the undamped natural frequency, in rad/s.
- $\zeta$ is the dimensionless damping ratio.
- $0<\zeta<1$: complex conjugate poles, oscillation, and step overshoot.
- $\zeta=1$: a repeated real pole; the fastest nonoscillatory **standard second-order step response for fixed $\omega_n$**, zero initial state, and no zeros.
- $\zeta>1$: two negative real poles; a slower nonoscillatory response under the same assumptions.

Run this cell to draw the damping comparison directly in the notebook. The orange shaded area highlights overshoot above the target; all curves use the same natural frequency.

In [ ]:
if VISUALIZE:
    fig, ax = plt.subplots(figsize=(9, 4.8))
    times = np.linspace(0, 5, 1501)
    for label, zeta, color in [("Underdamped", 0.35, ORANGE),
                               ("Critically damped", 1.0, TEAL),
                               ("Overdamped", 1.8, BLUE)]:
        t, response = step(([9.], [1., 6 * zeta, 9.]), T=times)
        ax.plot(t, response, color=color, label=f"{label}  ·  ζ = {zeta:g}")
        if zeta < 1:
            ax.fill_between(t, 1, response, where=response > 1, color=color, alpha=0.12)
            peak = np.argmax(response)
            ax.scatter(t[peak], response[peak], color=color, zorder=3)
            ax.annotate("31% overshoot", (t[peak], response[peak]), xytext=(2.4, 1.36),
                        arrowprops={"arrowstyle": "->", "color": color}, color=color)
    ax.axhline(1, color=GRAY, ls="--", lw=1)
    ax.set(xlabel="Time (s)", ylabel="Normalized position", xlim=(0, 5), ylim=(0, 1.55),
           title="SECOND-ORDER RESPONSE  /  Same natural frequency: 3 rad/s")
    ax.legend(loc="lower right", frameon=False)
    plt.show()


The comparison is computed from the normalized transfer function, rather than drawn by eye:

$$
\frac{\Theta(s)}{R(s)}=
\frac{\omega_n^2}{s^2+2\zeta\omega_n s+\omega_n^2}.
$$

For $0<\zeta<1$, its fractional peak overshoot is

$$
M_p=\exp\!\left(-\frac{\pi\zeta}{\sqrt{1-\zeta^2}}\right),
\qquad
\%\mathrm{OS}=100M_p.
$$

For $\zeta=0.35$, this predicts about 31% overshoot. These formulas assume no saturation and continuous feedback. A sampled or force-limited controller can differ.

### Compare three controllers on the same rotor

Set $r=1$ rad and $\omega_n=3$ rad/s. All three designs have $k_p=9$; their derivative gains are 0.1, 4.0, and 8.8 N m s/rad. A negative computed $k_d$ would cancel some physical damping; do not silently clamp it and claim the same design.

In [ ]:
motor_runs = {}
for name, zeta in [("underdamped", 0.35), ("critical", 1.0), ("overdamped", 1.8)]:
    kp, kd = 9.0, 2 * zeta * 3.0 - 2.0
    controller = lambda m, d, kp=kp, kd=kd: [kp * (1.0 - d.qpos[0]) - kd * d.qvel[0]]
    trace = rollout(motor, [0., 0.], controller, duration=DURATION, label=f"zeta = {zeta:g}")
    motor_runs[name] = trace
    show_run(trace, "rotor", "motor-" + name, [("Angle", "rad"), ("Angular velocity", "rad/s")])
    print(name, "peak overshoot (%):", 100 * max(0, trace.x[:, 0].max() - 1))

**Try it:** double $\omega_n$, recompute both gains, and compare required peak torque. Faster poles are not free: effort, noise sensitivity, sampling, and unmodeled dynamics can become limiting factors.

## 4. Stability, controllability, and pole placement

### Stability refers to an equilibrium

For $\dot{\mathbf{x}}=A\mathbf{x}$, the origin is asymptotically stable exactly when all eigenvalues of $A$ have strictly negative real parts. A positive real part implies instability. Eigenvalues on the imaginary axis require more care: defective Jordan blocks can cause polynomial growth. The unforced rotor has a continuum of stationary angles, not asymptotic convergence to one chosen angle.

For a nonlinear system, eigenvalues of the Jacobian describe **local** behavior near a verified equilibrium. Strictly negative real parts imply local asymptotic stability; any strictly positive real part implies instability. Zero real parts make this test inconclusive.

### Can this input influence every mode?

For an $n$-state LTI model, form the controllability matrix:

$$
\mathcal{C}=\begin{bmatrix}B&AB&A^2B&\cdots&A^{n-1}B\end{bmatrix}.
$$

Full rank $n$ permits arbitrary pole assignment by unrestricted full-state feedback. **Stabilizability** is weaker: uncontrollable modes are allowed if they are already asymptotically stable. The continuous-time PBH condition is

$$
\operatorname{rank}\begin{bmatrix}\lambda I-A&B\end{bmatrix}=n
\quad\text{for every eigenvalue with }\operatorname{Re}(\lambda)\ge0.
$$

Rank decisions are numerical and depend on scaling and tolerance. Inspect singular values if a model is poorly conditioned. Here our small examples have well-separated rank decisions.

To make the distinction concrete, the next cell examines a system whose input drives only its first state. It is not a simulation, so no additional mechanical model is needed.

Here is the full controllability calculation used throughout the practice. `place` refuses full pole assignment if that matrix is rank deficient, then calls SciPy with the $A-BK$ sign convention.

In [ ]:
def controllability(A, B):
    return np.hstack([np.linalg.matrix_power(A, k) @ B for k in range(len(A))])

def place(A, B, poles):
    if np.linalg.matrix_rank(controllability(A, B)) != len(A):
        raise ValueError('Full pole assignment requires controllability')
    return place_poles(A, B, poles).gain_matrix

In [ ]:
A_example, B_example = np.diag([-1., -2.]), np.array([[1.], [0.]])
print("Controllability rank:", np.linalg.matrix_rank(controllability(A_example, B_example)))
# The uncontrollable second mode is stable: stabilizable, but not controllable.
A_unstable = np.diag([-1., 2.])
print("PBH rank for lambda=+2:",
      np.linalg.matrix_rank(np.c_[2 * np.eye(2) - A_unstable, B_example]))
# Changing the unactuated mode to +2 makes stabilization impossible with this B.

### Place poles around a valid reference equilibrium

If $(\mathbf{x}_\star,\mathbf{u}_\star)$ is an equilibrium, choose

$$
\mathbf{u}=\mathbf{u}_\star-K(\mathbf{x}-\mathbf{x}_\star),
\qquad
\delta\dot{\mathbf{x}}=(A-BK)\delta\mathbf{x}.
$$

`place` first checks controllability, then calls SciPy's `place_poles`. Always inspect the achieved eigenvalues of $A-BK$. Requested poles do not guarantee that a sampled, saturated, noisy, or nonlinear realization behaves as the continuous linear design predicts. This practice assumes **full state measurement**; output-only feedback would need an observer or a different design.

The feedback function below implements this equation for the zero equilibrium input used in our examples. The linear simulator propagates the plant with an exact zero-order hold, then updates feedback at the next sample. This makes the comparison use the same timing and saturation as MuJoCo.

In [ ]:
def feedback(K, target=None):
    return lambda m, d: -K @ (state(m, d) - (np.zeros(m.nq + m.nv) if target is None else target))

def linear_rollout(A, B, initial, K, *, dt=0.002, duration=8.0, limit=np.inf, target=None):
    """Exact plant propagation with zero-order-held sampled feedback, matching rollout."""
    Ad, Bd, _, _, _ = cont2discrete((A, B, np.eye(len(A)), np.zeros((len(A), B.shape[1]))), dt)
    x = np.array(initial, float)
    target = np.zeros_like(x) if target is None else np.asarray(target)
    rows = [x.copy()]
    for _ in range(round(duration / dt)):
        u = np.clip(-K @ (x - target), -limit, limit)
        x = Ad @ x + Bd @ u
        rows.append(x.copy())
    return np.array(rows)

## 5. A flexible drive with two inertias

A rigid motor model can hide important motion. Introduce a load inertia connected to the motor through a torsional spring. Torque acts only on the motor; the spring transfers it to the load.



MuJoCo implements the spring with a **fixed tendon** whose scalar length is $\theta_l-\theta_m$. With zero rest length and stiffness $k$, its energy is $\tfrac12 k(\theta_l-\theta_m)^2$. The displayed bridge is a schematic coupling, not a rigid kinematic constraint. The two discs remain independent hinge bodies.

In [ ]:
def flexible_drive(*, load_inertia=1., motor_inertia=1., load_damping=2., motor_damping=2., stiffness=200., dt=0.002):
    bodies=[]
    for name,x,J,b,color in [('load',-.65,load_inertia,load_damping,'.75 .36 .1 1'),('motor',.65,motor_inertia,motor_damping,'.0784 .3608 .5961 1')]:
        bodies.append(f'''<body name="{name}" pos="{x} 0 .2"><joint name="{name}_angle" axis="0 0 1" damping="{b}"/>
          <inertial pos="0 0 0" mass="8" diaginertia="{.6*J} {.6*J} {J}"/>
          <geom type="cylinder" size=".4 .06" rgba="{color}"/>
          <geom type="capsule" fromto="0 0 .07 .35 0 .07" size=".025" rgba="1 1 1 1"/>
        </body>''')
    tendon=f'''<tendon><fixed name="torsion" stiffness="{stiffness}" springlength="0">
      <joint joint="load_angle" coef="1"/><joint joint="motor_angle" coef="-1"/>
    </fixed></tendon>'''
    return mujoco.MjModel.from_xml_string(xml_model(''.join(bodies),dt=dt,tendon=tendon,
        actuator='<motor joint="motor_angle" gear="1"/>',
        static='<geom type="capsule" fromto="-.25 0 .2 .25 0 .2" size=".025" rgba=".15 .48 .47 1"/>'))

In [ ]:
flex = flexible_drive(dt=DT)
preview(flex, np.zeros(4), "flexible")

### Derive the coupled equations

The load and motor equations are

$$
J_l\ddot\theta_l+b_l\dot\theta_l+k(\theta_l-\theta_m)=0,
$$

$$
J_m\ddot\theta_m+b_m\dot\theta_m+k(\theta_m-\theta_l)=u.
$$

Use $J_l=J_m=1$ kg m$^2$, $b_l=b_m=2$ N m s/rad, and $k=200$ N m/rad. Our MuJoCo state groups positions before velocities:

$$
\mathbf{x}=\begin{bmatrix}\theta_l&\theta_m&\dot\theta_l&\dot\theta_m\end{bmatrix}^{\mathsf T},
$$

$$
A=\begin{bmatrix}
0&0&1&0\\0&0&0&1\\
-k/J_l&k/J_l&-b_l/J_l&0\\
k/J_m&-k/J_m&0&-b_m/J_m
\end{bmatrix},\qquad
B=\begin{bmatrix}0\\0\\0\\1/J_m\end{bmatrix}.
$$

The source notebook interleaved each angle and its velocity. Reusing its gain row without reordering the state would control the wrong variables. The same warning applies to matrices, initial states, and plot labels.

### Compare motor-side PD with full-state feedback

Both controllers target $\theta_l=\theta_m=1$ rad at rest. The spring is then untwisted and zero input is an equilibrium. Motor-side PD uses only $\theta_m$ and $\dot\theta_m$; full-state feedback also uses load motion.

First verify MuJoCo's local model against the exact matrix, then check controllability and place poles at $-1,-1.1,-1.5,-2$ s$^{-1}$. `linearize` perturbs state/input, calls `mj_forward`, and central-differences the continuous derivative; it does **not** return a discrete step Jacobian.

The following implementation first checks that the chosen state and input are an equilibrium. It then perturbs each state and input independently to estimate the continuous Jacobians.

In [ ]:
def linearize(model, equilibrium=None, input_equilibrium=None, eps=1e-6):
    n = model.nq + model.nv
    x = np.zeros(n) if equilibrium is None else np.asarray(equilibrium, dtype=float)
    u = (np.zeros(model.nu) if input_equilibrium is None
         else np.atleast_1d(input_equilibrium).astype(float))
    if np.linalg.norm(derivative(model, x, u)) > 1e-8:
        raise ValueError("Linearize about a verified equilibrium")
    A = np.empty((n, n))
    B = np.empty((n, model.nu))
    for i, direction in enumerate(np.eye(n)):
        plus = derivative(model, x + eps * direction, u)
        minus = derivative(model, x - eps * direction, u)
        A[:, i] = (plus - minus) / (2 * eps)
    for i, direction in enumerate(np.eye(model.nu)):
        plus = derivative(model, x, u + eps * direction)
        minus = derivative(model, x, u - eps * direction)
        B[:, i] = (plus - minus) / (2 * eps)
    return A, B

In [ ]:
A_f, B_f = linearize(flex)
A_expected = np.array([[0, 0, 1, 0], [0, 0, 0, 1],
                       [-200, 200, -2, 0], [200, -200, 0, -2.]])
np.testing.assert_allclose(A_f, A_expected, atol=1e-8)
print("Controllability rank:", np.linalg.matrix_rank(controllability(A_f, B_f)))
K_f = place(A_f, B_f, [-1., -1.1, -1.5, -2.])
print("Gain:", K_f)
print("Achieved poles:", np.linalg.eigvals(A_f - B_f @ K_f))
target_f = np.array([1., 1., 0., 0.])
flex_labels = [("Load angle", "rad"), ("Motor angle", "rad"),
               ("Load velocity", "rad/s"), ("Motor velocity", "rad/s")]

Run both controllers from rest. The full-state result is also checked against the exact sampled linear model. This plant is linear in its unwrapped joint angles, so close agreement should persist beyond tiny perturbations.

In [ ]:
flex_pd = rollout(flex, np.zeros(4), feedback(np.array([[0., 10., 0., 10.]]), target_f),
                  duration=DURATION, label="Motor-side PD")
show_run(flex_pd, "flexible", "flexible-pd", flex_labels)
flex_poles = rollout(flex, np.zeros(4), feedback(K_f, target_f),
                     duration=DURATION, label="Full-state pole placement")
flex_linear = linear_rollout(A_f, B_f, np.zeros(4), K_f, dt=DT,
                             duration=DURATION, target=target_f)
show_run(flex_poles, "flexible", "flexible-poles", flex_labels, flex_linear)
print("Maximum state discrepancy:", np.max(np.abs(flex_poles.x - flex_linear)))

**Read the result:** both designs can settle for these parameters. Pole placement is not automatically faster or less demanding than PD; the chosen poles specify a different transient. Compare load error, relative twist, peak torque, and sensitivity to spring stiffness before choosing a design.

## 6. Nonlinear equilibria and local models

The original control lesson also studies a nonmechanical two-state vector field:

$$
\dot a=a-a^3+2ab,\qquad
\dot b=-b+\tfrac12 ab.
$$

Treat $a,b$ as dimensionless coordinates and time in normalized units. Solving both right-hand sides equal to zero gives four equilibria:

$$
(a_\star,b_\star)\in\{(0,0),(1,0),(-1,0),(2,1.5)\}.
$$

Its Jacobian is

$$
J_f(a,b)=\begin{bmatrix}
1-3a^2+2b&2a\\
b/2&-1+a/2
\end{bmatrix}.
$$

The phase portrait will be drawn from the equations below, together with the simulated trajectory.

### Make an honest MuJoCo realization

An arbitrary first-order ODE is not the natural motion of a passive rigid body. To visualize it with MuJoCo, use a unit point mass with two orthogonal slide joints and active force control. Let its positions represent $\mathbf{q}=[a,b]^\mathsf{T}$ and apply

$$
\mathbf{u}=J_f(\mathbf{q})\mathbf{v}
+\alpha\bigl(f(\mathbf{q})-\mathbf{v}\bigr),\qquad \alpha=20.
$$

Since $\ddot{\mathbf{q}}=\mathbf{u}$ in these normalized units, the mismatch $\mathbf{e}=\mathbf{v}-f(\mathbf{q})$ obeys

$$
\dot{\mathbf{e}}=-\alpha\mathbf{e}.
$$

With $\mathbf{v}(0)=f(\mathbf{q}(0))$, the continuous system follows the original vector field exactly. The MuJoCo realization has **four states**, with two additional stable mismatch modes. Our sampled force introduces a small discretization error; verify it rather than claiming exact equivalence. The extra forces are part of an engineered realization, not a property of the original ODE.

Load the planar marker. Small fixed markers indicate equilibria; they exert no contact forces.

These definitions implement the vector field, its Jacobian, the active force law, and the two-slide-joint MuJoCo body. The field and controller are visible together so you can verify the mismatch derivation directly.

In [ ]:
EQUILIBRIA = np.array([[0., 0.], [1., 0.], [-1., 0.], [2., 1.5]])

def field(q):
    a, b = q
    return np.array([a - a ** 3 + 2 * a * b, -b + 0.5 * a * b])

def field_jacobian(q):
    a, b = q
    return np.array([[1 - 3 * a * a + 2 * b, 2 * a], [0.5 * b, -1 + 0.5 * a]])

def field_controller(model, data, gain=20.0):
    return field_jacobian(data.qpos) @ data.qvel + gain * (field(data.qpos) - data.qvel)

def vector_field_model(*, dt=.002):
    static=''.join(f'<geom type="sphere" pos="{x} {y} .06" size=".06" rgba=".15 .48 .47 1"/>' for x,y in EQUILIBRIA)
    body='''<body name="state_marker" pos="0 0 .15"><joint name="x1" type="slide" axis="1 0 0"/>
      <joint name="x2" type="slide" axis="0 1 0"/><inertial pos="0 0 0" mass="1" diaginertia=".01 .01 .01"/>
      <geom type="sphere" size=".1"/></body>'''
    return mujoco.MjModel.from_xml_string(xml_model(body,dt=dt,static=static,
        actuator='<motor joint="x1"/><motor joint="x2"/>'))

In [ ]:
field_model = vector_field_model(dt=DT)
q0 = np.array([-2., -1.])
x0_field = np.r_[q0, field(q0)]
preview(field_model, x0_field, "field")
for point in EQUILIBRIA:
    print(point, "f =", field(point), "local eigenvalues =",
          np.linalg.eigvals(field_jacobian(point)))

The equilibria $(1,0)$ and $(-1,0)$ are stable nodes; $(0,0)$ and $(2,1.5)$ are saddles. This is local information. The phase portrait helps us see trajectories and attraction regions that eigenvalues alone cannot describe.

### Compare the realization with the original ODE

Start at $(-2,-1)$, integrate the two-state ODE independently with `solve_ivp`, and compare its position trajectory with the actively controlled MuJoCo mass. Plot and render the MuJoCo motion. Then measure the manifold mismatch; decreasing `DT` should reduce this sampled-control error.

In [ ]:
field_run = rollout(field_model, x0_field, field_controller,
                    duration=DURATION, label="MuJoCo realization")
show_run(field_run, "field", "nonlinear-field",
         [("a", "normalized"), ("b", "normalized"),
          ("a rate", "normalized/s"), ("b rate", "normalized/s")])
ode = solve_ivp(lambda t, q: field(q), [0., DURATION], q0,
                t_eval=field_run.time, rtol=1e-10, atol=1e-12)
assert ode.success
mismatch = field_run.x[:, 2:] - np.array([field(q) for q in field_run.x[:, :2]])
print("Maximum position discrepancy:", np.max(np.abs(field_run.x[:, :2] - ode.y.T)))
print("Maximum velocity-manifold mismatch:", np.max(np.abs(mismatch)))
if VISUALIZE:
    fig, ax = plt.subplots(figsize=(9, 5.5))
    a_grid, b_grid = np.linspace(-2.5, 2.5, 41), np.linspace(-1.5, 2.2, 41)
    aa, bb = np.meshgrid(a_grid, b_grid)
    ax.streamplot(a_grid, b_grid, aa - aa**3 + 2*aa*bb, -bb + 0.5*aa*bb,
                  color="#c3d1de", density=1, linewidth=0.7, arrowsize=0.8)
    for a, b in EQUILIBRIA:
        stable = a in [-1, 1]
        ax.scatter(a, b, color=TEAL if stable else ORANGE,
                   marker="o" if stable else "x", s=55, zorder=4)
        ax.annotate(f"({a:g}, {b:g})", (a, b), xytext=(8, 10), textcoords="offset points")
    ax.set(xlim=(-2.5, 2.7), ylim=(-1.5, 2.2), title="NONLINEAR FLOW  /  Local equilibria and a trajectory")
    ax.plot(ode.y[0], ode.y[1], "--", label="Original first-order ODE")
    ax.plot(field_run.x[:, 0], field_run.x[:, 1], label="MuJoCo realization")
    ax.set(xlabel="a (normalized)", ylabel="b (normalized)")
    ax.legend()
    plt.show()


## 7. Cart-pole stabilization

The cart-pole adds gravity, nonlinear geometry, and underactuation. A horizontal force moves the cart; there is no actuator at the pendulum hinge. The controller must move the support to balance the pole.



Load the system at a small positive tilt before deriving its model. The ideal rail is unbounded and contact-free. The camera follows the cart during videos; **use the position plot to measure translation**, not its position on screen.

The cart is a translating body and the pole is its hinged child. The negative $y$ hinge axis makes positive angle lean left. An explicit bob inertia keeps the analytic and MuJoCo models consistent.

In [ ]:
def cart_pole(*, cart_mass=2., pole_mass=.5, length=.3, bob_inertia=.0001, dt=.002):
    # Negative y hinge: positive theta leans to the LEFT, as in the supplied notebook.
    body=f'''<body name="cart" pos="0 0 .2"><joint name="cart_position" type="slide" axis="1 0 0"/>
      <inertial pos="0 0 0" mass="{cart_mass}" diaginertia=".02 .02 .02"/>
      <geom type="box" size=".15 .1 .08"/>
      <body name="pole" pos="0 0 .08"><joint name="pole_angle" axis="0 -1 0"/>
        <inertial pos="0 0 {length}" mass="{pole_mass}" diaginertia="{bob_inertia} {bob_inertia} {bob_inertia}"/>
        <geom type="capsule" fromto="0 0 0 0 0 {length}" size=".012" rgba=".15 .48 .47 1"/>
        <geom type="sphere" pos="0 0 {length}" size=".05" rgba=".75 .36 .1 1"/>
      </body></body>'''
    return mujoco.MjModel.from_xml_string(xml_model(body,dt=dt,gravity='0 0 -9.81',
        actuator='<motor joint="cart_position" gear="1"/>',
        static='<geom type="capsule" fromto="-3 0 .1 3 0 .1" size=".015" rgba=".4 .47 .53 1"/>'))

In [ ]:
cart = cart_pole(cart_mass=2., pole_mass=0.5, length=0.3,
                 bob_inertia=0.0001, dt=DT)
preview(cart, [0., 0.12, 0., 0.], "cart")

### Coordinates and physical assumptions

- $p$: horizontal cart position, positive to the right, in m.
- $\theta$: pole angle from **upright**, positive when leaning left, in rad.
- $M=2$ kg: cart mass; $m=0.5$ kg: bob mass.
- $L=0.3$ m: hinge-to-bob-center distance; $g=9.81$ m/s$^2$.
- $J=10^{-4}$ kg m$^2$: small bob inertia about its center. The rod is visual and contributes no inertia because the body's inertia is specified explicitly.

The source notebook uses an ideal point bob. We retain a small positive bob inertia in MuJoCo and include it in the analytic equations; setting $J\to0$ recovers the source's model.

With $H=mL^2+J$, the nonlinear equations are

$$
(M+m)\ddot p-mL\cos\theta\,\ddot\theta
+mL\sin\theta\,\dot\theta^2=u,
$$

$$
H\ddot\theta-mL\cos\theta\,\ddot p-mgL\sin\theta=0.
$$

The negative coupling signs follow from the bob's horizontal position $p-L\sin\theta$. Changing the angle convention changes the signs of the matrices and the feedback gain.

Use the MuJoCo order

$$
\mathbf{x}=\begin{bmatrix}p&\theta&\dot p&\dot\theta\end{bmatrix}^{\mathsf T}.
$$

The source notebook used $[\theta,\dot\theta,p,\dot p]^\mathsf T$. To convert one source state, reorder its entries as `[2, 0, 3, 1]`; do not copy its gain vector unchanged.

### Linearize around upright rest

The target is $\mathbf{x}_\star=\mathbf{0}$, $u_\star=0$. Define

$$
\Delta=(M+m)H-(mL)^2.
$$

The continuous linearization is

$$
A=\begin{bmatrix}
0&0&1&0\\0&0&0&1\\
0&m^2gL^2/\Delta&0&0\\
0&(M+m)mgL/\Delta&0&0
\end{bmatrix},\qquad
B=\begin{bmatrix}0\\0\\H/\Delta\\mL/\Delta\end{bmatrix}.
$$

The positive gravity terms express the instability of upright balance. The input can nevertheless control all four linearized states.

Compare the analytic matrices with central differences of MuJoCo's forward dynamics. A finite-difference agreement checks this particular model and convention; it is not evidence that the approximation works far from upright.

In [ ]:
def cart_matrices(M=2.0, m=0.5, L=0.3, J=0.0001, g=9.81):
    H = m * L * L + J
    determinant = (M + m) * H - (m * L) ** 2
    A = np.array([[0, 0, 1, 0], [0, 0, 0, 1], [0, m * m * g * L * L / determinant, 0, 0], [0, (M + m) * m * g * L / determinant, 0, 0.0]])
    B = np.array([[0.0], [0.0], [H / determinant], [m * L / determinant]])
    return (A, B)

In [ ]:
A_c, B_c = linearize(cart)
A_analytic, B_analytic = cart_matrices()
np.testing.assert_allclose(A_c, A_analytic, atol=1e-7)
np.testing.assert_allclose(B_c, B_analytic, atol=1e-7)
print("Open-loop poles:", np.linalg.eigvals(A_c))
print("Controllability rank:", np.linalg.matrix_rank(controllability(A_c, B_c)))

### Place the poles and check the sampled controller

Choose distinct poles $\{-1,-2,-3,-4\}$ s$^{-1}$, as in the original cart-pole lesson. Compute $K$ for $u=-K\mathbf{x}$, then check both the continuous poles and the spectral radius of the sampled closed loop.

$$
\mathbf{x}_{k+1}=(A_d-B_dK)\mathbf{x}_k,
\qquad \rho(A_d-B_dK)<1.
$$

This discrete check applies to the unsaturated linear plant. It is necessary for that local sampled design, not a guarantee of global nonlinear stabilization.

In [ ]:
K_c = place(A_c, B_c, [-1., -2., -3., -4.])
print("K =", K_c)
print("Continuous closed-loop poles:", np.linalg.eigvals(A_c - B_c @ K_c))
Ad, Bd, _, _, _ = cont2discrete((A_c, B_c, np.eye(4), np.zeros((4, 1))), DT)
print("Sampled spectral radius:", max(abs(np.linalg.eigvals(Ad - Bd @ K_c))))
cart_labels = [("Cart position", "m"), ("Pole angle", "rad"),
               ("Cart velocity", "m/s"), ("Pole velocity", "rad/s")]

### Small tilt: verify local stabilization

Start at $\theta(0)=0.12$ rad with everything else zero, and limit force to $\pm30$ N. Use the same saturation in the linear comparison. Inspect cart position, angle, and both velocities on separate axes with their proper units.

In [ ]:
small_initial = [0., 0.12, 0., 0.]
cart_small = rollout(cart, small_initial, feedback(K_c), duration=DURATION,
                     limit=30., label="Nonlinear MuJoCo")
small_linear = linear_rollout(A_c, B_c, small_initial, K_c, dt=DT,
                              duration=DURATION, limit=30.)
show_run(cart_small, "cart", "cart-small", cart_labels, small_linear)
print("Final state:", cart_small.x[-1])

### Large tilt: expose the limits of the design

Now use the source notebook's $1.2$ rad initial tilt (about $69^\circ$). Keep the gains and force limit unchanged. Show the first 1.5 seconds so the loss of balance remains readable; this is an intentionally unsuccessful experiment, not a swing-up controller.

In [ ]:
large_initial = [0., 1.2, 0., 0.]
cart_large = rollout(cart, large_initial, feedback(K_c), duration=1.5,
                     limit=30., label="Nonlinear MuJoCo")
large_linear = linear_rollout(A_c, B_c, large_initial, K_c, dt=DT,
                              duration=1.5, limit=30.)
show_run(cart_large, "cart", "cart-large", cart_labels, large_linear)
if VISUALIZE:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(cart_large.time, cart_large.requested[:, 0], label="Requested force")
    ax.plot(cart_large.time, cart_large.applied[:, 0], label="Applied force")
    ax.set(xlabel="Time (s)", ylabel="Force (N)")
    ax.legend()
    plt.show()


**Interpretation:** two assumptions fail together: small-angle geometry and unconstrained feedback. At the initial large tilt, the requested force already exceeds 30 N. Both the saturated linear model and the nonlinear plant can depart from the intended response. To separate the effects, first increase the limit while keeping the initial state fixed, then reduce the angle while keeping the limit fixed. Re-render each new simulation. A swing-up strategy and a switching rule would be a later topic.

## 8. Tasks, checks, and documentation

### What to submit

1. **Model check:** derive the spring and flexible-drive matrices in the stated coordinate order. Report a numerical derivative check against MuJoCo.
2. **Response design:** predict and measure motor overshoot for $\zeta=0.35$. Compare critical and overdamped settling with the same $\omega_n$ and explain the torque tradeoff when increasing $\omega_n$.
3. **Controllability:** report rank and achieved poles for the flexible drive and cart-pole. Explain why controllability does not remove actuator limits.
4. **Nonlinear check:** classify all four vector-field equilibria. Compare the MuJoCo realization with `solve_ivp`, and halve the control period to test convergence of the mismatch.
5. **Cart-pole comparison:** show small- and large-tilt trajectories, force histories, and a MuJoCo visualization for each. Change one of tilt, force limit, or pole locations at a time.

Submit the notebook or a concise report, configuration, model/package versions, labeled figures, and videos. Keep both successful and unsuccessful experiments. Use the course rubric: correctness/checks 40%, controlled comparison 30%, interpretation 20%, reproducibility 10%.

### Baseline checks

- Spring equilibrium: 0 m without force, 0.4 m with a 2 N step.
- Unforced motor: final angle near 1.5 rad from $[1,1]^\mathsf T$.
- Motor with $\zeta=0.35$: approximately 31% overshoot; small sampled-control differences are expected.
- Flexible drive: controllability rank 4 and agreement with the sampled linear trajectory to well below $10^{-5}$ for the baseline.
- Vector field: stable nodes at $(\pm1,0)$, saddles at $(0,0)$ and $(2,1.5)$; the trajectory from $(-2,-1)$ approaches $(-1,0)$.
- Cart-pole: controllability rank 4; analytic and MuJoCo linearizations agree within $10^{-7}$. The small-tilt final state is close to zero after 10 s. The large-tilt experiment loses balance under the stated limit.

### Inspect and modify the models

Edit the constructor parameters or the MJCF definitions directly in the corresponding notebook cells. Recreate `MjModel` and `MjData` after changing a compiled model's structure. Recompute $A,B,K$ when changing the model used for controller design; keep the original gains only when deliberately testing mismatch.

Useful specification and API documentation:

- [MJCF overview and model structure](https://mujoco.readthedocs.io/en/3.13.0/modeling.html): bodies, joints, inertia, defaults, and compilation.
- [Joint stiffness and damping](https://mujoco.readthedocs.io/en/3.13.0/XMLreference.html#body-joint): passive forces used by the spring and rotor models.
- [Fixed tendons](https://mujoco.readthedocs.io/en/3.13.0/XMLreference.html#tendon-fixed): linear combinations of joint coordinates for the flexible coupling.
- [Motor actuators](https://mujoco.readthedocs.io/en/3.13.0/XMLreference.html#actuator-motor): control-to-force mapping and transmission gear.
- [Forward dynamics and simulation](https://mujoco.readthedocs.io/en/3.13.0/APIreference/APIfunctions.html#mj-forward): distinguish `mj_forward` from time integration with `mj_step`.
- [Model editing with MjSpec](https://mujoco.readthedocs.io/en/3.13.0/programming/modeledit.html): programmatic model construction and modification.
- [Python rendering and bindings](https://mujoco.readthedocs.io/en/3.13.0/python.html): renderer lifetime and offscreen contexts.
- [SciPy pole placement](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.place_poles.html): the sign convention is $A-BK$.
- [SciPy continuous-to-discrete conversion](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.cont2discrete.html): zero-order-hold propagation for sampled inputs.

### Source continuity and reproducibility

This practice adapts the supplied [control-basics notebook](https://colab.research.google.com/drive/1Ci2DXi5MOk0UpptgVkqE49_JLAfojPcd) and [cart-pole notebook](https://colab.research.google.com/drive/1F4FbgGSjZ-rxDbnQbZft2G6R_-MC18Ld). It retains their mass–spring–damper, torque-controlled motor, flexible two-inertia drive, nonlinear vector field, and cart-pole. State order, the direct-feedthrough dimension, sampled feedback, the finite bob inertia, and the nonlinear realization are made explicit here.

The diagrams follow a textbook schematic style: flat colored geometry, bold outlines, dashed guides, and italic mathematical symbols. All numerical plots are computed in notebook cells, not stored as figure assets. The [reference damping illustration](https://encyclopedia.pub/media/common/202210/mceclip0-63467f7db5b20.png) supplies the comparison idea; the replacement uses computed second-order responses. Every simulation embeds its MuJoCo video below its plotting cell. Videos and plots belong to executed notebook outputs, not to versioned course assets. Before committing the tutorial, clear its outputs. Edit this notebook directly; the website renders its Markdown and code cells without saved outputs.

For visual style, the schematics follow the [supplied cart-pole reference](https://drive.usercontent.google.com/download?id=1O1iM6MByfMyrO0txAMFb_miOqPNzFAvm&authuser=0). Our cart-pole drawing preserves the positive-left angle convention used in the equations and MJCF.

Continue with [practice 03: manipulator control](../../practices/p03-manipulator/README.md).